In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# import scipy.stats as stats
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder, TargetEncoder
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.compose import ColumnTransformer
# from sklearn.feature_selection import VarianceThreshold
from sklearn.base import clone
# from sklearn.metrics import root_mean_squared_error

from sklearn.linear_model import Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer

from prepare_data import prepare_data

In [ ]:
data = pd.read_csv('data/train.csv')
data

In [ ]:
# 1. Проверка распределения целевой переменной
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(data['SalePrice'], kde=True, ax=axes[0])
axes[0].set_title('Распределение SalePrice (Original)')

# Используем логарифм для сравнения
sns.histplot(np.log1p(data['SalePrice']), kde=True, ax=axes[1])
axes[1].set_title('Распределение SalePrice (Log-transformed)')

plt.tight_layout()
plt.show()

In [ ]:
data['SalePrice'].skew()

In [ ]:
y = np.log1p(data['SalePrice'])
X = prepare_data(data.drop('SalePrice',axis=1))

In [ ]:
y.skew()

In [ ]:
# 2. Поиск выбросов в ключевых признаках
plt.figure(figsize=(10, 6))
sns.scatterplot(x='GrLivArea', y='SalePrice', data=data)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 1. Удаление выбросов согласно рекомендациям автора датасета
X = X[X['GrLivArea'] < 4500]

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='GrLivArea', y='SalePrice', data=pd.concat([X,y]))
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#синхронизируем данные
y = y[X.index]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    shuffle=True,
)

In [ ]:
numeric_colums = X_train.select_dtypes(include=['number']).columns
categorical_colums = X_train.select_dtypes(include=['object', 'string']).columns

In [ ]:
fig = plt.figure()

fig.set_size_inches(16,10)

sns.heatmap(X[numeric_colums].corr(),
            xticklabels=numeric_colums,
            yticklabels=numeric_colums,
            cmap='BrBG',
            vmin=-1,
            vmax=1)

plt.show()

In [ ]:
# 3. Топ-10 признаков, коррелирующих с ценой
plt.figure(figsize=(10, 8))
correlations = X[numeric_colums].corrwith(y).sort_values(ascending=False)
sns.barplot(x=correlations[1:11].values, y=correlations[1:11].index, hue=correlations[1:11].index, palette='magma')
plt.show()

In [ ]:
low_card_cols = [
    col for col in categorical_colums
    if X_train[col].nunique() < 5
]

high_card_cols = [
    col for col in categorical_colums
    if X_train[col].nunique() >= 5
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    # ('variance_selector', VarianceThreshold(threshold=0.1))

])

categorical_imputer = SimpleImputer(strategy='constant', fill_value='missing')  

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', Pipeline([
            ('imputer', categorical_imputer),
            ('encoder', OneHotEncoder(
            drop='first',
            handle_unknown='ignore',
            sparse_output=False
        ))
        ]), low_card_cols),

        ('target', Pipeline([
            ('imputer', categorical_imputer),
            ('target',  TargetEncoder(
            target_type='continuous',
            random_state=42
        ))]), high_card_cols),

        ('numeric', numeric_transformer, numeric_colums)
    ],
    verbose_feature_names_out=False
)

preprocessor

In [ ]:
preview_preprocessor = clone(preprocessor)
preview_preprocessor.set_output(transform='pandas')

X_encoded_preview = preview_preprocessor.fit_transform(X, y)
X_encoded_preview.head()

In [ ]:
models = {
    'Lasso': (
        Lasso(),
        {
            'model__alpha': np.logspace(-4, 0, 20)
        }
    ),

    'Ridge': (
        Ridge(),
        {
            'model__alpha': np.logspace(-4, 0, 20)
        }
    ),

    'tree': (
        DecisionTreeRegressor(random_state=42),
        {
            # 'model__max_depth' : np.linspace(3, 10, 7, dtype=int),
            # 'model__min_samples_leaf': np.linspace(1, 5, 5, dtype=int),
            # 'model__max_leaf_nodes': np.linspace(5, 100, 20, dtype=int)

            'model__max_depth': [3, 5, 7],
            'model__min_samples_leaf': [1, 3, 5],
            'model__max_leaf_nodes': [3,5,7],

        }
    ),

    'forest': (
        RandomForestRegressor(random_state=42),
        {
            # 'model__n_estimators': [100],
            # 'model__max_depth': np.linspace(3, 10, 7, dtype=int),
            # 'model__min_samples_leaf': np.linspace(1, 5, 5, dtype=int),
            # 'model__max_leaf_nodes': np.linspace(5, 100, 20, dtype=int)

            'model__n_estimators': [100],
            'model__max_depth': [3, 5, 7],
            'model__min_samples_leaf': [1, 3, 5],
        }
    ),

    'extra_trees': (
        ExtraTreesRegressor(random_state=42),
        {
            # 'model__n_estimators': [100],
            # 'model__max_depth': np.linspace(3, 10, 7, dtype=int),
            # 'model__min_samples_leaf': np.linspace(1, 5, 5, dtype=int),
            # 'model__max_leaf_nodes': np.linspace(5, 100, 20, dtype=int)

            'model__n_estimators': [100],
            'model__max_depth': [3, 5, 7],
            'model__min_samples_leaf': [1, 3, 5],
        }
    ),

    'gradient_boosting': (
        GradientBoostingRegressor(random_state=42),
        {
            # 'model__n_estimators': [100],
            # 'model__max_depth': np.linspace(3, 10, 7, dtype=int),
            # 'model__min_samples_leaf': np.linspace(1, 5, 5, dtype=int),
            # 'model__max_leaf_nodes': np.linspace(5, 100, 20, dtype=int)

            'model__n_estimators': [100],
            'model__max_depth': [3, 5, 7],
            'model__min_samples_leaf': [1, 3, 5],
        }
    ),
    'XGBClassifier' :(
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=1,
            verbosity=0,
        ),
        {
            'model__n_estimators': [50, 100, 200, 300],
            'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'model__max_depth': [3, 5, 7, 9],
            'model__subsample': [0.6, 0.8, 1.0],
            'model__colsample_bytree': [0.6, 0.8, 1.0],
            'model__gamma': [0, 0.1, 0.2],
        },
    ),
    'LGBMClassifier':(
        LGBMClassifier(
            objective="binary",
            random_state=42,
            n_jobs=1,
            verbose=-1,
        ),
        {
            'model__n_estimators': [50, 100, 200, 300],
            'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'model__max_depth': [-1, 3, 5, 7, 9],
            'model__num_leaves': [20, 31, 40],
            'model__min_child_samples': [20, 30, 40],
            'model__subsample': [0.6, 0.8, 1.0],
            'model__colsample_bytree': [0.6, 0.8, 1.0],

        },
    ),
}

In [ ]:
results = []

# Запуск цикла обучения с поиском по сетке
for name, (model, params) in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])

    grid = GridSearchCV(pipe, params, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
    grid.fit(X_train, y_train)

    results.append({
        'model': name,
        'best_cv_score': grid.best_score_,
        'test_score': grid.score(X_test, y_test),
        'best_params': grid.best_params_,
        'best_estimator': grid.best_estimator_
    })

print("Обучение завершено. Рассчитываем финальные метрики...")

In [ ]:
params_catboost = {
            'model__iterations': [100, 200, 300, 500, 1000],
            'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'model__depth': [3, 5, 7, 9],
            'model__l2_leaf_reg': [1, 3, 5, 10],
            'model__random_strength': [0.1, 0.5, 1, 2],
            'model__border_count': [32, 64, 128, 254],
        }

cat_features_list = X.select_dtypes(include=['object', 'string']).columns.tolist()

pipe = Pipeline([
        ('model',  CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="Accuracy",
            random_seed=42,
            thread_count=1,
            verbose=0,
            allow_writing_files=False,
        )),
    ])


grid = GridSearchCV(pipe, params_catboost, cv=5, scoring='accuracy')

grid.fit(X_train, y_train, model__cat_features = cat_features_list,)

results.append({
        'model': 'CatBoostClassifier',
        'best_cv_score': grid.best_score_,
        'test_score': grid.score(X_test, y_test),
        'best_params': grid.best_params_,
        'best_estimator': grid.best_estimator_
    })

In [ ]:
# 

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    'test_score',
    ascending=False
)

results_df['best_cv_score_original_scale'] = np.expm1(results_df['best_cv_score'])
results_df['test_score_original_scale'] = np.expm1(results_df['test_score'])

print(results_df[['model', 'best_cv_score', 'test_score', 'best_params','best_cv_score_original_scale','test_score_original_scale']])

In [ ]:
best_model = results_df.sort_values(
    'test_score_original_scale',
    ascending=False
).iloc[0]['best_estimator']
best_model

In [ ]:
test_data = pd.read_csv('data/test.csv')
y_pred = best_model.predict(prepare_data(test_data))

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": y_pred
})

submission_path = 'data/submission.csv'
submission.to_csv(submission_path, index=False)